In [1]:
import numpy as np
import matplotlib.pyplot as plt
from Utilities_copy import extractor
import uproot
import awkward as ak    

x_MH25=extractor("/home/riccardo/Tesi/Cartella_Analisi_Dati/Dati/Tprime_tAq_1800_MH25_LH_2017.root", "Events")


file=uproot.open("/home/riccardo/Tesi/Cartella_Analisi_Dati/Dati/Tprime_tAq_1800_MH25_LH_2017.root")
tree=file["Events"]
booleans= tree.arrays(["FatJet_isMatchedWithA"], library="ak")
booleanas=tree.arrays(["FatJet_isMatchedWith2BHadrons"], library="ak")
Fatjet_isMatchedWithA= booleans["FatJet_isMatchedWithA"]
Fatjet_isMatchedWith2BHadrons= booleanas["FatJet_isMatchedWith2BHadrons"]
#Filtriamo i dati

mask = (ak.flatten(Fatjet_isMatchedWithA) == 1) & (ak.flatten(Fatjet_isMatchedWith2BHadrons) == 1)
x_filtered = x_MH25[mask]


/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/cppyy/__init__.py:374: UserWarning: CPyCppyy API not found (tried: /home/riccardo/anaconda3/envs/rootnev/include/site/python3.14); set CPPYY_API_PATH envar to the 'CPyCppyy' API directory to fix
  warnings.warn("CPyCppyy API not found (tried: %s); "
/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/awkward/_nplikes/array_module.py:289: RuntimeWarning: invalid value encountered in divide
  return impl(*broadcasted_args, **(kwargs or {}))


In [2]:
from scipy.special import voigt_profile
from iminuit import Minuit
from iminuit.cost import LeastSquares

x_plot=list(x_filtered)
x_plot.sort()
x_easy=[x for x in x_plot if 0 < x < 100]

def voigt2(x, norm, mu, sigma, gamma, norm2, mu2, sigma2, gamma2):
    return voigt_profile(x-mu, sigma, gamma) * norm + norm2*voigt_profile(x-mu2, sigma2, gamma2)

bin_counts, bin_edges = np.histogram(x_easy, bins=50)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = bin_edges[1] - bin_edges[0]
bin_densities = bin_counts / (len(x_easy) * bin_width)  # Densità normalizzata
yerr=np.sqrt(bin_counts) / (len(x_easy) * bin_width) # Errore standard per i dati binned

ls_voigt=LeastSquares(bin_centers, bin_densities, yerr, model=voigt2)

m_voigt=Minuit(ls_voigt,  norm=1, mu=25, sigma=5, gamma=1, norm2=1, mu2=25, sigma2=5, gamma2=0.001)
m_voigt.limits["mu"]= (0, 100)
m_voigt.limits["sigma"]= (0.1, 20)
m_voigt.limits["gamma"]= (0.01, 10)
m_voigt.fixed["gamma2"]= True

m_voigt.migrad()


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 719.2 (χ²/ndof = 16.7)     │              Nfcn = 513              │
│ EDM = 9.87e-05 (Goal: 0.0002)    │            time = 0.4 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬────────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name   │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼────────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ norm   │   0.682   │   0.014   │            │            │         │         │       │
│ 1 │ mu     │   24.49   │   0.05    │            │            │    0    │   100   │       │
│ 2 │ sigma  │   2.995   │   0.024   │            │            │   0.1   │   20    │       │
│ 3 │ gamma  │   0.207   │   0.011   │            │            │  0.01   │   10    │       │
│ 4 │ norm2  │   0.308   │   0.014   │            │            │         │         │       │
│ 5 │ mu2    │   27.18   │   0.05    │            │            │         │         │       │
│ 6 │ sigma2 │   1.70    │   0.04    │            │            │         │         │       │
│ 7 │ gamma2 │  1.00e-3  │  0.01e-3  │            │            │         │         │  yes  │
└───┴────────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌────────┬─────────────────────────────────────────────────────────────────────────┐
│        │     norm       mu    sigma    gamma    norm2      mu2   sigma2   gamma2 │
├────────┼─────────────────────────────────────────────────────────────────────────┤
│   norm │ 0.000211  0.62e-3 -0.05e-3 -0.02e-3 -0.20e-3  0.47e-3 -0.47e-3        0 │
│     mu │  0.62e-3  0.00243        0 -0.09e-3 -0.62e-3   0.0010  -0.0014   0.0000 │
│  sigma │ -0.05e-3        0 0.000585 -0.15e-3  0.05e-3  -0.6e-3   0.1e-3        0 │
│  gamma │ -0.02e-3 -0.09e-3 -0.15e-3 0.000128  0.02e-3  0.02e-3  0.05e-3        0 │
│  norm2 │ -0.20e-3 -0.62e-3  0.05e-3  0.02e-3 0.000203 -0.46e-3  0.47e-3        0 │
│    mu2 │  0.47e-3   0.0010  -0.6e-3  0.02e-3 -0.46e-3  0.00225  -0.0013   0.0000 │
│ sigma2 │ -0.47e-3  -0.0014   0.1e-3  0.05e-3  0.47e-3  -0.0013  0.00143   0.0000 │
│ gamma2 │        0   0.0000        0        0        0   0.0000   0.0000        0 │
└────────┴─────────────────────────────────────────────────────────────────────────┘

In [3]:
fit_MH25_values={}
fit_MH25_errors={}

fit_values={'MH25': fit_MH25_values,}
fit_errors={'MH25_errors': fit_MH25_errors}



for param in m_voigt.parameters:
    fit_MH25_values[param] = m_voigt.values[param]

for error in m_voigt.parameters:    #Qui non ho capito come fa a capire che deve estarre gli errori 
    fit_MH25_errors[error] = m_voigt.errors[error]

print(fit_MH25_values)
print(fit_MH25_errors)

import json
#QUi sono andato di metodo oragutang, ho deciso di voler fare 2 file separati peer errori e valori 
#Ho tenuto lo stesso quello con tutti i valori, casomai cambiassi idea

with open("fit_results.json", "r") as f:
    results=json.load(f)

with open("fit_values.json", "r") as g:
    values=json.load(g) 

with open("fit_errors.json", "r") as h:
    errors=json.load(h)


results["MH25"]=fit_MH25_values
results["MH25_errors"]=fit_MH25_errors

with open("fit_results.json", "w") as f:
    json.dump(results, f, indent=1)

values["MH25"]=fit_MH25_values
with open("fit_values.json", "w") as g:
    json.dump(values, g, indent=1)  

errors["MH25_errors"]=fit_MH25_errors
with open("fit_errors.json", "w") as h:
    json.dump(errors, h, indent=1)  

{'norm': 0.682020534094887, 'mu': 24.4948584181444, 'sigma': 2.9947645616897876, 'gamma': 0.20662519986136063, 'norm2': 0.3077370942677364, 'mu2': 27.176364848190314, 'sigma2': 1.6981769202210721, 'gamma2': 0.001}
{'norm': 0.014517856701713962, 'mu': 0.0492763335929407, 'sigma': 0.024183142182699013, 'gamma': 0.0113196536781634, 'norm2': 0.014245516920382425, 'mu2': 0.0474420613999034, 'sigma2': 0.037821387871113665, 'gamma2': 1e-05}
